# ♻️ MISSÃO RECOMEÇO · FASE 2
### Simulado de Visão 1 · Trilha de Visão Computacional · as fotos da Fase 1, agora inteiras

---

> **"Na primeira vez vocês mediram catorze coisas de cada foto. Eu perguntei se dava para a
> máquina olhar a foto como eu olho. Disseram que agora dá."**
>
> — Rosa Bezerra, presidente da Cooperativa Recomeço

## 📨 A Cooperativa Recomeço voltou

> São **as mesmas fotos** da Fase 1: papel, papelão, plástico, vidro e metal, fotografados
> um a um sobre um fundo. A diferença é que, desta vez, elas chegam **inteiras**, com o
> tamanho original, e não mais reduzidas a 48 pixels.
>
> E vocês não vão mais escrever o código que mede cor e brilho. Vão usar um modelo que já
> olhou mais de um milhão de fotos antes de conhecer a cooperativa, e ensinar a ele só os
> nossos cinco materiais.
>
> Duas advertências de quem trabalha na esteira. **Papel e papelão continuam sendo a dupla
> que mais confunde.** E **as fotos foram todas tiradas no mesmo fundo**: se o modelo
> aprender o fundo em vez do material, a esteira de verdade vai mostrar.
>
> — Rosa Bezerra

---

*Dados: TrashNet (Thung & Yang, 2016), sem a classe "lixo": as mesmas 2.390 fotos da Missão
Recomeço da Fase 1, em resolução cheia. 240 fotos rotuladas viraram o teste oculto, como na
Fase 1.*

## 🎯 A sua missão


Ensinar um modelo **pré-treinado** a reconhecer o material de cada foto, escolhendo entre
**EfficientNet-B0**, **ResNet-50** e **ViT-B/16**, em dois estágios de treino — e mostrar,
com o mapa de atenção, que ele acerta **olhando o material**.


### O que muda da Fase 1 para cá

| Na Fase 1 | Na Fase 2 |
|:---|:---|
| Fotos de 48×48 e **catorze medidas calculadas à mão** | A **foto inteira** em 224×224: o modelo descobre sozinho o que olhar |
| O modelo nascia vazio | O modelo chega **sabendo enxergar** (pré-treinado em ~1 milhão de fotos) |
| `.fit` e pronto | **Dois estágios** de treino e quatro botões |
| A métrica e a matriz | + **onde o modelo olhou**: o objeto, ou o fundo? |

### 🗺️ O mapa da missão

| Bloco | O que vocês fazem | Como |
|:---|:---|:---|
| **1 · EXPLORAR** | Contar as fotos, medir o piso, ver o que a resolução e o aumento fazem | ✏️ **1 lacuna** |
| **2 · CONFIGURAR** | Escolher o modelo e girar os quatro botões | Só arrastando |
| **3 · MODELAR** | Células A (preparo), B (dois estágios) e C (avaliação) | ✏️ **11 lacunas** + Zona Livre |
| **4 · AVALIAR** | Curvas, matriz, mapa de atenção, grupos | Só clicando |

> ⚠️ **Erro nº 1 da fase: esquecer a GPU.** `Ambiente de execução` → `Alterar o tipo de
> ambiente de execução` → **T4 GPU**. Sem ela o código roda igual, sem mensagem nenhuma, e
> demora umas vinte vezes mais. A primeira célula avisa se a GPU não estiver ligada.
>
> **Erro nº 2: a sessão cai e leva tudo.** O ponto de verificação guarda a melhor época na
> memória; para não perder o modelo se a sessão cair, a célula C também salva um arquivo.
>
> 📎 **Simulado de preparação.** Formato dos cadernos da Fase 2, com dados públicos. O
> gabarito do teste fica com o corretor (`simulados/corrigir.py`), nunca dentro do caderno.

In [ ]:
#@title 🛠️ Preparação: abrindo as fotos { display-mode: "form" }
# fotos.zip fica na mesma pasta deste caderno (no Colab, a célula pede o envio).
SEMENTE = 42
import pathlib, zipfile, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import torch

if not pathlib.Path("fotos").exists():
    if not pathlib.Path("fotos.zip").exists():
        from google.colab import files
        print("Envie o arquivo fotos.zip:")
        files.upload()
    zipfile.ZipFile("fotos.zip").extractall("fotos")
PASTA_TREINO = pathlib.Path("fotos/treino")
FOTOS_TESTE = sorted(pathlib.Path("fotos/teste").glob("*.jpg"))
IDS_TESTE = np.array([int(f.stem) for f in FOTOS_TESTE])
N_TESTE = len(FOTOS_TESTE)

# ImageFolder numera as classes em ordem alfabética das pastas.
CLASSES = sorted(p.name for p in PASTA_TREINO.iterdir() if p.is_dir())
EMOJI = {'papel': '📄', 'papelao': '📦', 'plastico': '🧴', 'vidro': '🍾', 'metal': '🥫'}
arquivos = {k: sorted((PASTA_TREINO / k).glob("*.jpg")) for k in CLASSES}

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Ambiente carregado.\n")
print(f"  Conjunto público (rotulado) ....... {sum(map(len, arquivos.values())):>5} fotos")
print(f"  Conjunto de teste (sem rótulo) .... {N_TESTE:>5} fotos  [tamanho fixo]")
print(f"  Dispositivo ....................... {DISPOSITIVO}")
if DISPOSITIVO == "cpu":
    print("\n[!] ERRO Nº 1: sem GPU. O código roda igual, sem mensagem nenhuma, e cada época")
    print("    leva minutos em vez de segundos. Liguem a T4 antes de chegar no Bloco 3.")

---
# 🔍 BLOCO 1 · EXPLORAR
## *Olhem as fotos antes de treinar qualquer coisa*


> **Rosa:** *"Na Fase 1 a foto chegava borrada. Comparem com o que vocês vão ver agora."*


Três perguntas para este bloco: **quantas fotos tem cada classe** (e quanto vale um modelo que
não aprende nada), **o que a resolução apaga**, e **o que o aumento de dados faz** com uma foto.

In [ ]:
# ✏️ SUA VEZ · Quantas fotos, e o piso
contagem = pd.Series({k: len(v) for k, v in arquivos.items()})
y_publico = np.concatenate([[i] * len(arquivos[k]) for i, k in enumerate(CLASSES)])
chute = np.full_like(y_publico, contagem.values.argmax())
# LACUNA 1 ▸ que média do F1 é a métrica da fase?
piso = f1_score(y_publico, chute, average=____)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar([f"{EMOJI[k]} {k}" for k in CLASSES], contagem.values, color="#00B8D4")
for i, v in enumerate(contagem.values):
    ax.text(i, v, str(v), ha="center", va="bottom")
ax.set_title("Fotos por classe no conjunto público", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
print(f"O PISO: responder '{CLASSES[contagem.values.argmax()]}' para tudo dá "
      f"{accuracy_score(y_publico, chute)*100:.1f}% de acurácia e {piso*100:.1f}% de F1-macro.")

In [ ]:
#@title 📷 As fotos de cada classe { display-mode: "form" }
classe_para_ver = "vidro"  #@param ["metal", "papel", "papelao", "plastico", "vidro"]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, f in zip(axes.ravel(), arquivos[classe_para_ver][:8]):
    ax.imshow(Image.open(f)); ax.axis("off")
fig.suptitle(f"{EMOJI[classe_para_ver]}  {classe_para_ver.upper()}", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()
print("Reparem na luz, no enquadramento e no FUNDO. Se cada classe foi fotografada num")
print("lugar diferente, o modelo pode aprender o lugar em vez do objeto (Bloco 4).")

In [ ]:
#@title 🔍 De 48 pixels para 224: o que a resolução apaga { display-mode: "form" }
foto_numero = 0  #@param {type:"slider", min:0, max:30, step:1}
_f = arquivos[CLASSES[foto_numero % len(CLASSES)]][foto_numero]
_im = Image.open(_f).convert("RGB")
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, r in zip(axes, [24, 48, 96, 224]):
    ax.imshow(_im.resize((r, r), Image.BILINEAR).resize((224, 224), Image.NEAREST))
    ax.set_title(f"{r}×{r} · {r*r/1000:.1f} mil pixels", fontweight="bold"); ax.axis("off")
plt.tight_layout(); plt.show()
print("Na Fase 1 a foto chegava em 48×48. Os três modelos desta fase foram treinados em 224:")
print("entregar menos joga informação fora; entregar mais custa memória sem ganhar quase nada.")

In [ ]:
#@title 🎲 Uma foto vira quatro: o aumento de dados { display-mode: "form" }
recorte_minimo = 0.6  #@param {type:"slider", min:0.3, max:1.0, step:0.05}
brilho = 0.3  #@param {type:"slider", min:0.0, max:0.8, step:0.05}
espelhar = True  #@param {type:"boolean"}
from torchvision.transforms import RandomResizedCrop, RandomHorizontalFlip, ColorJitter, Compose
_aum = Compose([RandomResizedCrop(224, scale=(recorte_minimo, 1.0))]
               + ([RandomHorizontalFlip()] if espelhar else []) + [ColorJitter(brightness=brilho)])
fig, axes = plt.subplots(1, 5, figsize=(15, 3.4))
axes[0].imshow(_im.resize((224, 224))); axes[0].set_title("original", fontweight="bold")
for k, ax in enumerate(axes[1:], 1):
    ax.imshow(_aum(_im)); ax.set_title(f"época {k}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("A cada época o modelo vê uma versão diferente da mesma foto e não consegue decorá-la.")
print("A REGRA DE OURO: o aumento vai SÓ no treino. A validação é o mundo como ele é.")

---
# ⚙️ BLOCO 2 · CONFIGURAR
## *Três modelos que já sabem enxergar*

| Conjunto | Tamanho | Vocês têm os rótulos? | Para que serve |
|:---|:---|:---|:---|
| **Público** | 2.150 fotos | Sim | Treinar e validar |
| **Teste** | **240 (fixo)** | Não | Comparação final (`simulados/corrigir.py`) |

### 📗 As três fichas

| | ⚡ EfficientNet-B0 | 🏛️ ResNet-50 | 🧩 ViT-B/16 |
|:---|:---|:---|:---|
| **O que é** | Convolução desenhada para o melhor resultado pelo menor custo | A convolução mais conhecida, com atalhos internos que estabilizam o treino | Corta a foto em quadradinhos de 16 px e trata cada um como uma palavra |
| **Construtor** | `efficientnet_b0` | `resnet50` | `vit_b_16` |
| **Camada de saída** | `modelo.classifier[1]` | `modelo.fc` | `modelo.heads.head` |
| **Entram nela** | 1280 | 2048 | 768 |
| **Taxa do estágio 2** | `1e-4` | `1e-4` | `1e-5` (o mais sensível) |
| **Lote** | 32 | 32 | 16 |
| **A pegadinha** | A saída **não** se chama `fc`. Copiar a linha da ResNet não quebra: cria um atributo novo e treina a saída errada | A preparação da imagem vem com os pesos: nunca escrevam à mão | Entrada **exatamente** 224×224 |
| **O que esperar** | A mais barata; resultado parecido com a ResNet | A escolha segura; o mapa de atenção funciona muito bem | ~4× mais cara; pode perder para as CNNs (e isso vira parágrafo de relatório) |

### O treino acontece em dois estágios

| Estágio | O que treina | Passo | Épocas |
|:---|:---|:---|:---|
| **0 · como chega** | nada: o tronco sabe enxergar, mas a saída responde **mil classes** que não são as de vocês | — | — |
| **1 · sondagem** | só a **camada de saída nova**; o tronco fica congelado | `1e-3` | 3 a 5 |
| **2 · ajuste fino** | libera o **último bloco** do tronco | `1e-4` (ViT `1e-5`) | 10 a 15 |

Descongelar o modelo inteiro com poucas fotos é receita de decoreba: ele apaga o que sabia e
decora o que vocês mostraram.

### Quatro botões, e só quatro

| Botão | Faixa de trabalho | Fora dela |
|:---|:---|:---|
| **Taxa** | `1e-4` (ViT `1e-5`) no estágio 2 | `1e-2` atropela o pré-treino (curva serrilha); `1e-6` mal sai do lugar |
| **Lote** | 32 (ViT 16) | grande demais não cabe na memória da GPU |
| **Épocas** | 3–5 + 10–15 | demais: decoreba; a parada antecipada protege |
| **Aumento** | leve | nenhum: decora as fotos; forte demais: ensina coisas que não existem |

> Os valores iniciais abaixo são **deliberadamente modestos** (e um deles está fora da ficha).
> `usar_fracao_das_fotos` serve para uma primeira volta rápida: comecem com 0,3 para ver o
> caderno rodar inteiro, e voltem a 1,0 para o resultado de verdade.

In [ ]:
#@title 🎛️ Parâmetros do experimento { display-mode: "form" }
modelo_escolhido = "EfficientNet-B0"  #@param ["EfficientNet-B0", "ResNet-50", "ViT-B/16"]
tamanho_da_validacao = 20  #@param {type:"slider", min:10, max:40, step:5}
usar_fracao_das_fotos = 1.0  #@param {type:"slider", min:0.05, max:1.0, step:0.05}
lote = 32  #@param [8, 16, 32, 64] {type:"raw"}
aumento = "nenhum"  #@param ["nenhum", "leve", "forte"]
taxa_estagio1 = 1e-3  #@param [1e-2, 1e-3, 1e-4] {type:"raw"}
epocas_estagio1 = 3  #@param {type:"slider", min:1, max:8, step:1}
taxa_estagio2 = 1e-2  #@param [1e-2, 1e-3, 1e-4, 1e-5, 1e-6] {type:"raw"}
epocas_estagio2 = 6  #@param {type:"slider", min:0, max:20, step:1}
paciencia = 3  #@param {type:"slider", min:1, max:8, step:1}

FICHA = {"EfficientNet-B0": (1e-4, 32), "ResNet-50": (1e-4, 32), "ViT-B/16": (1e-5, 16)}
print("PARÂMETROS DEFINIDOS")
print(f"   Modelo .............. {modelo_escolhido}")
print(f"   Estágio 1 ........... taxa {taxa_estagio1:g} · {epocas_estagio1} épocas (só a saída)")
print(f"   Estágio 2 ........... taxa {taxa_estagio2:g} · {epocas_estagio2} épocas (último bloco)")
print(f"   Lote · aumento ...... {lote} · {aumento}   |   fração das fotos: {usar_fracao_das_fotos:.0%}")
_taxa_ficha, _lote_ficha = FICHA[modelo_escolhido]
if taxa_estagio2 > _taxa_ficha * 5:
    print(f"\n[!] A ficha da {modelo_escolhido} pede {_taxa_ficha:g} no estágio 2. Um passo desse")
    print("    tamanho num modelo pré-treinado não ajusta: atropela o que ele já sabia.")
if lote > _lote_ficha:
    print(f"\n[!] A ficha pede lote {_lote_ficha}. Lote maior pode estourar a memória da GPU.")

---
# ✏️ BLOCO 3 · MODELAR
## *A prepara, B treina, C avalia*

Três erros desta trilha **não dão mensagem nenhuma**, e a verificação deste bloco procura os três:

1. **O aumento na validação.** A validação recebe o preparo do treino, com recorte ao acaso: a
   nota passa a depender do sorteio. A verificação passa a mesma foto duas vezes pela fila da
   validação; se sair diferente, o aumento vazou.
2. **A saída com o nome de outro modelo.** Em Python, atribuir a um nome que não existe cria
   um atributo novo, que o modelo ignora. O treino roda e a saída de mil classes continua lá.
3. **Esquecer o modo de avaliação.** Sem `modelo.eval()`, o dropout continua sorteando
   neurônios na hora de prever.

> ⚠️ Preencham **todas** as lacunas, inclusive as dos modelos que não escolheram.

In [ ]:
#@title 🧰 Peças prontas: congelar, liberar, treinar e o mapa de atenção { display-mode: "form" }
import time, copy
import torch
from torch import nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision.transforms import (Compose, Resize, CenterCrop, RandomResizedCrop,
                                    RandomHorizontalFlip, ColorJitter, ToTensor, Normalize)
from torchvision.models import (efficientnet_b0, EfficientNet_B0_Weights, resnet50,
                                ResNet50_Weights, vit_b_16, ViT_B_16_Weights)

torch.manual_seed(SEMENTE)


def saida_do(modelo, nome):
    """A camada de saída de verdade de cada arquitetura (é ela que o estágio 1 treina)."""
    return {"ResNet-50": lambda: modelo.fc, "EfficientNet-B0": lambda: modelo.classifier[1],
            "ViT-B/16": lambda: modelo.heads.head}[nome]()


def ultimo_bloco(modelo, nome):
    """O bloco do tronco que o estágio 2 libera: onde moram as características mais específicas."""
    return {"ResNet-50": lambda: [modelo.layer4],
            "EfficientNet-B0": lambda: [modelo.features[-2], modelo.features[-1]],
            "ViT-B/16": lambda: [modelo.encoder.layers[-1], modelo.encoder.ln]}[nome]()


def congelar_tudo_menos_a_saida(modelo, nome):
    for p in modelo.parameters():
        p.requires_grad = False
    for p in saida_do(modelo, nome).parameters():
        p.requires_grad = True


def liberar_ultimo_bloco(modelo, nome):
    for bloco in ultimo_bloco(modelo, nome):
        for p in bloco.parameters():
            p.requires_grad = True


treinaveis = lambda modelo: [p for p in modelo.parameters() if p.requires_grad]
contar = lambda ps: sum(p.numel() for p in ps)


def treinar_estagio(modelo, dados_tr, dados_va, otimizador, agendador, perda, epocas,
                    paciencia, ponto_de_verificacao, estado):
    """Laço de treino de um estágio. 'estado' guarda o melhor modelo entre os dois estágios."""
    sem_melhorar = 0
    for epoca in range(epocas):
        modelo.train()
        soma, n = 0.0, 0
        for fotos, resposta in dados_tr:
            fotos, resposta = fotos.to(DISPOSITIVO), resposta.to(DISPOSITIVO)
            otimizador.zero_grad()              # 1 · apaga o passo anterior
            erro = perda(modelo(fotos), resposta)  # 2 e 3 · passagem para a frente e o erro
            erro.backward()                     # 4 · propaga o erro para trás
            otimizador.step()                   # 5 · anda
            soma += erro.item() * len(resposta); n += len(resposta)
        if agendador is not None:
            agendador.step()
        modelo.eval()
        with torch.no_grad():
            s, m = 0.0, 0
            for fotos, resposta in dados_va:
                fotos, resposta = fotos.to(DISPOSITIVO), resposta.to(DISPOSITIVO)
                s += perda(modelo(fotos), resposta).item() * len(resposta); m += len(resposta)
        estado["treino"].append(soma / n); estado["validacao"].append(s / m)
        ep = len(estado["treino"])
        print(f"   época {ep:>2} · perda treino {soma/n:.3f} · validação {s/m:.3f}", flush=True)
        if s / m < estado["melhor_perda"]:
            estado["melhor_perda"], estado["melhor_epoca"], sem_melhorar = s / m, ep, 0
            if ponto_de_verificacao:
                estado["melhor_pesos"] = copy.deepcopy(modelo.state_dict())
        else:
            sem_melhorar += 1
            if sem_melhorar >= paciencia:
                print("   parada antecipada: a validação parou de melhorar.")
                break


@torch.no_grad()
def prever(modelo, dados):
    """Probabilidades de cada classe. Atenção: NÃO liga o modo de avaliação sozinho."""
    probs, alvos = [], []
    for fotos, resposta in dados:
        probs.append(torch.softmax(modelo(fotos.to(DISPOSITIVO)), dim=1).cpu())
        alvos.append(resposta)
    return torch.cat(probs).numpy(), torch.cat(alvos).numpy()


def mapa_de_atencao(modelo, nome, foto, classe):
    """Grad-CAM: onde o modelo olhou para dar a nota da 'classe'."""
    camada = {"ResNet-50": lambda: modelo.layer4, "EfficientNet-B0": lambda: modelo.features[-1],
              "ViT-B/16": lambda: modelo.encoder.layers[-1]}[nome]()
    guarda = {}

    def ida(m, i, o):
        guarda["a"] = o
        o.register_hook(lambda g: guarda.__setitem__("g", g))
    h = camada.register_forward_hook(ida)
    x = foto.unsqueeze(0).to(DISPOSITIVO).requires_grad_(True)
    modelo.zero_grad()
    modelo(x)[0, classe].backward()
    h.remove()
    a, g = guarda["a"].detach(), guarda["g"].detach()
    if nome == "ViT-B/16":  # 197 pedaços = 1 de classe + 14×14 quadradinhos
        a = a[:, 1:].reshape(1, 14, 14, -1).permute(0, 3, 1, 2)
        g = g[:, 1:].reshape(1, 14, 14, -1).permute(0, 3, 1, 2)
    cam = torch.relu((g.mean(dim=(2, 3), keepdim=True) * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=foto.shape[-2:], mode="bilinear")[0, 0]
    return (cam / (cam.max() + 1e-8)).cpu().numpy()


print("Peças prontas.")

In [ ]:
# ═══ CÉLULA A · preparo ═══
PESOS = {"EfficientNet-B0": EfficientNet_B0_Weights.IMAGENET1K_V1,
         "ResNet-50": ResNet50_Weights.IMAGENET1K_V2,
         "ViT-B/16": ViT_B_16_Weights.IMAGENET1K_V1}[modelo_escolhido]

# LACUNA 2 ▸ a preparação da validação vem JUNTO com os pesos, nunca escrita à mão.
#            Qual método dos pesos devolve essa preparação?
preparo_val = PESOS.____()
media, desvio = preparo_val.mean, preparo_val.std

# No treino entra o aumento: a mesma foto, sempre diferente.
AUMENTO = {"nenhum": [Resize(256), CenterCrop(224)],
           "leve": [RandomResizedCrop(224, scale=(0.7, 1.0)), RandomHorizontalFlip()],
           "forte": [RandomResizedCrop(224, scale=(0.4, 1.0)), RandomHorizontalFlip(),
                     ColorJitter(0.4, 0.4, 0.4)]}[aumento]
preparo_tr = Compose(AUMENTO + [ToTensor(), Normalize(media, desvio)])

# LACUNAS 3 e 4 ▸ o aumento vai SÓ no treino; a validação é o mundo como ele é.
base_tr = ImageFolder(PASTA_TREINO, transform=____)
base_va = ImageFolder(PASTA_TREINO, transform=____)

# Separação estratificada (a mesma foto nunca está dos dois lados).
_alvos = np.array(base_tr.targets)
_idx = np.arange(len(_alvos))
if usar_fracao_das_fotos < 1:
    _idx, _ = train_test_split(_idx, train_size=usar_fracao_das_fotos, stratify=_alvos,
                               random_state=SEMENTE)
idx_tr, idx_va = train_test_split(_idx, test_size=tamanho_da_validacao/100,
                                  stratify=_alvos[_idx], random_state=SEMENTE)
_trab = 2 if DISPOSITIVO == "cuda" else 0
dados_tr = DataLoader(Subset(base_tr, idx_tr), batch_size=lote, shuffle=True, num_workers=_trab)
dados_va = DataLoader(Subset(base_va, idx_va), batch_size=lote, shuffle=False, num_workers=_trab)
print(f"Treino: {len(idx_tr)} fotos  |  Validação: {len(idx_va)} fotos  |  classes: {base_tr.classes}")

In [ ]:
# ═══ CÉLULA B · treino em dois estágios ═══
CONSTRUTOR = {"EfficientNet-B0": efficientnet_b0, "ResNet-50": resnet50, "ViT-B/16": vit_b_16}
modelo = CONSTRUTOR[modelo_escolhido](weights=PESOS)
n_classes = len(CLASSES)

# LACUNAS 5, 6 e 7 ▸ cada modelo guarda a saída com um nome diferente (ver as fichas).
#                   Errar aqui NÃO dá erro: cria um atributo novo e treina a saída errada.
if modelo_escolhido == "ResNet-50":
    modelo.____ = nn.Linear(2048, n_classes)
elif modelo_escolhido == "EfficientNet-B0":
    modelo.____ = nn.Linear(1280, n_classes)
else:
    modelo.____ = nn.Linear(768, n_classes)
modelo = modelo.to(DISPOSITIVO)

# ═══ ZONA LIVRE ═══  (vazia = correto e mediano; um ajuste por vez)
suavizacao_dos_rotulos = 0.0   # label_smoothing da perda: 0.0 a 0.2
decaimento_de_pesos = 0.01     # weight_decay do AdamW
# ══════════════════
perda = nn.CrossEntropyLoss(label_smoothing=suavizacao_dos_rotulos)
estado = {"treino": [], "validacao": [], "melhor_perda": float("inf"),
          "melhor_epoca": 0, "melhor_pesos": None}
inicio = time.time()

# LACUNA 8 ▸ guardar a melhor época (e não a última)?  True ou False
ponto_de_verificacao = ____

# ESTÁGIO 1 · sondagem: o tronco fica congelado e só a saída nova aprende.
congelar_tudo_menos_a_saida(modelo, modelo_escolhido)
# LACUNAS 9 e 10 ▸ quem move os pesos, e com qual passo neste estágio?
otimizador = ____(treinaveis(modelo), lr=____, weight_decay=decaimento_de_pesos)
print(f"ESTÁGIO 1 · treinando {contar(treinaveis(modelo)):,} pesos")
treinar_estagio(modelo, dados_tr, dados_va, otimizador, None, perda, epocas_estagio1,
                paciencia, ponto_de_verificacao, estado)
fim_estagio1 = len(estado["treino"])

# ESTÁGIO 2 · ajuste fino: libera o último bloco, com passo menor.
if epocas_estagio2 > 0:
    liberar_ultimo_bloco(modelo, modelo_escolhido)
    # LACUNA 11 ▸ o passo do estágio 2 (o da ficha, bem menor que o do estágio 1)
    otimizador = AdamW(treinaveis(modelo), lr=____, weight_decay=decaimento_de_pesos)
    agendador = CosineAnnealingLR(otimizador, T_max=epocas_estagio2)
    print(f"ESTÁGIO 2 · treinando {contar(treinaveis(modelo)):,} pesos")
    treinar_estagio(modelo, dados_tr, dados_va, otimizador, agendador, perda,
                    epocas_estagio2, paciencia, ponto_de_verificacao, estado)

if ponto_de_verificacao and estado["melhor_pesos"] is not None:
    modelo.load_state_dict(estado["melhor_pesos"])
tempo_de_treino = time.time() - inicio
print(f"\nTreino concluído em {tempo_de_treino/60:.1f} min · melhor época: {estado['melhor_epoca']}")

In [ ]:
# ═══ CÉLULA C · avaliação ═══  (só uma linha para completar; o resto é para ler)
# LACUNA 12 ▸ antes de prever: desligar o dropout e o modo de treino.
modelo.____()
prob_va, y_va = prever(modelo, dados_va)
previsoes = prob_va.argmax(axis=1)
f1 = f1_score(y_va, previsoes, average="macro")
acuracia = accuracy_score(y_va, previsoes)
torch.save(modelo.state_dict(), "modelo_melhor_epoca.pt")  # se a sessão cair, o modelo fica

if "registro" not in dir():
    registro = []
registro.append({"modelo": modelo_escolhido, "taxa_est2": taxa_estagio2, "aumento": aumento,
                 "fracao": usar_fracao_das_fotos, "melhor_epoca": estado["melhor_epoca"],
                 "f1_macro": round(f1, 4), "minutos": round(tempo_de_treino / 60, 1)})
print("="*60)
print("            DESEMPENHO NO CONJUNTO DE VALIDAÇÃO")
print("="*60)
print(f"  Piso (chute preguiçoso) ............... {piso*100:5.1f}%")
print(f"  ⭐ F1-MACRO (métrica oficial) .......... {f1*100:5.1f}%")
print(f"     Acurácia (métrica de apoio) ......... {acuracia*100:5.1f}%")
print("="*60)
a, s_, i_ = (0.9, 0.85, 0.75)
if   f1 >= a:  print("  Desempenho alto. Agora confiram ONDE o modelo olhou (Bloco 4).")
elif f1 >= s_: print("  Desempenho sólido. Há margem nos quatro botões e na Zona Livre.")
elif f1 >= i_: print("  Desempenho intermediário. Leiam as curvas: o formato diz qual botão mexer.")
else:          print("  Desempenho baixo. Confiram a taxa do estágio 2 e a camada de saída.")

In [ ]:
#@title 🔎 Verificação das lacunas (os três erros que não dão erro) { display-mode: "form" }
_estava_em_treino = modelo.training   # como a célula C deixou o modelo
erros = []
try:
    if not np.isclose(piso, f1_score(y_publico, chute, average="macro")):
        erros.append("LACUNA 1: o piso não é o F1-macro do chute preguiçoso.")
except NameError:
    erros.append("LACUNA 1: a variável 'piso' não existe.")
if type(preparo_val).__name__ != "ImageClassification":
    erros.append("LACUNA 2: a preparação da validação precisa vir dos pesos (PESOS.transforms()).")
_i = int(idx_va[0])
if not torch.equal(base_va[_i][0], base_va[_i][0]):
    erros.append("LACUNA 4: A MESMA FOTO SAIU DIFERENTE DUAS VEZES NA VALIDAÇÃO. O aumento vazou "
                 "para a validação: a nota passa a depender do sorteio de cada rodada.")
if base_tr.transform is base_va.transform:
    erros.append("LACUNAS 3 e 4: treino e validação estão com o MESMO preparo.")
modelo.eval()
with torch.no_grad():
    _saida = modelo(base_va[_i][0].unsqueeze(0).to(DISPOSITIVO))
if _saida.shape[1] != n_classes:
    erros.append(f"LACUNAS 5 a 7: o modelo responde {_saida.shape[1]} classes, não {n_classes}. A "
                 "saída nova foi pendurada num nome que esta arquitetura não usa: o treino rodou "
                 "inteiro sobre a saída de mil classes do pré-treino.")
for _nome in ("fc", "classifier", "heads"):
    _attr = getattr(modelo, _nome, None)
    if isinstance(_attr, nn.Linear) and _attr is not saida_do(modelo, modelo_escolhido):
        erros.append(f"LACUNAS 5 a 7: existe um modelo.{_nome} solto que esta arquitetura ignora.")
if ponto_de_verificacao is not True:
    erros.append("LACUNA 8: sem ponto de verificação, a entrega é o modelo da última época.")
if type(otimizador).__name__ != "AdamW":
    erros.append(f"LACUNA 9: esperado AdamW, obtido {type(otimizador).__name__}.")
if _estava_em_treino:
    erros.append("LACUNA 12: a célula C previu com o modelo em modo de TREINO: o dropout "
                 "continuou sorteando neurônios e a nota muda a cada leitura.")

if erros:
    print("VERIFICAÇÃO REPROVADA. Pontos a revisar:\n")
    for e in erros: print("   [!] " + e + "\n")
    # O caderno para aqui: seguir com uma lacuna errada é deixar a nota mentir.
    raise RuntimeError("VERIFICAÇÃO REPROVADA: corrijam os pontos acima antes de seguir.")
else:
    print("VERIFICAÇÃO APROVADA\n")
    print("   Preparo dos pesos na validação, aumento só no treino, saída certa, melhor época.")
    if taxa_estagio2 > FICHA[modelo_escolhido][0] * 5:
        print(f"\n   Atenção: a taxa do estágio 2 ({taxa_estagio2:g}) está acima da ficha.")

---
# 📈 BLOCO 4 · AVALIAR
## *O modelo olhou para o objeto, ou para o fundo?*

| Formato das curvas | O que significa numa trilha de imagem |
|:---|:---|
| As duas descem, distância pequena | **Como deve ser.** A entrega é a melhor época, não a última |
| Treino desce, validação sobe | **Decoreba.** Com poucas fotos e milhões de pesos é a regra: aumento, dois estágios, parada antecipada |
| Validação muito abaixo do treino o tempo todo | **Validação fácil demais**: procurem fotos repetidas entre treino e validação |
| As duas serrilham e não descem | **Passo grande demais**: o passo apagou o pré-treino. Diminuam a taxa primeiro |

In [ ]:
#@title 📉 As curvas dos dois estágios { display-mode: "form" }
tr, va = np.array(estado["treino"]), np.array(estado["validacao"])
ep = np.arange(1, len(tr) + 1)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(ep, tr, "-o", color="#3B5BDB", lw=2, label="treino")
ax.plot(ep, va, "--o", color="#E6177A", lw=2, label="validação")
ax.axvspan(.5, fim_estagio1 + .5, color="#EEE", label="estágio 1 (só a saída)")
ax.axvline(estado["melhor_epoca"], color="#2F9E44", ls=":", lw=2,
           label=f"melhor época ({estado['melhor_epoca']})")
ax.set_xlabel("épocas →"); ax.set_ylabel("perda"); ax.legend()
ax.set_title("Perda de treino e de validação", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

if len(va) > 2 and np.mean(np.diff(va) > 0) > .45 and va[-1] > va.min() * 1.1:
    diag = "PASSO GRANDE DEMAIS: a validação serrilha. Diminuam a taxa do estágio 2."
elif va[-1] > va.min() * 1.1 and tr[-1] < tr[estado["melhor_epoca"] - 1]:
    diag = "DECOREBA: treino descendo, validação subindo depois da melhor época."
elif (va < tr * .6).mean() > .7:
    diag = "VALIDAÇÃO FÁCIL DEMAIS: procurem fotos repetidas entre treino e validação."
else:
    diag = "COMO DEVE SER: as duas descem e a distância fica pequena."
print(f"DIAGNÓSTICO: {diag}")
print("Um pouco de validação abaixo do treino é normal: o treino roda com aumento e dropout.")

In [ ]:
#@title 🧩 O modelo não erra ao acaso: a matriz { display-mode: "form" }
cm = confusion_matrix(y_va, previsoes, labels=range(n_classes))
prop = cm / cm.sum(axis=1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, m, titulo, fmt in [(axes[0], cm, "em contagem", "{:d}"),
                           (axes[1], prop, "em proporção da linha", "{:.0%}")]:
    ax.imshow(m, cmap="Purples")
    for i in range(n_classes):
        for j in range(n_classes):
            ax.text(j, i, fmt.format(m[i, j]), ha="center", va="center", fontsize=9,
                    color="white" if m[i, j] > m.max() * .6 else "black")
    ax.set_xticks(range(n_classes)); ax.set_xticklabels(CLASSES, rotation=30)
    ax.set_yticks(range(n_classes)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("o modelo disse"); ax.set_ylabel("a foto era"); ax.set_title(titulo, fontweight="bold")
plt.tight_layout(); plt.show()
_pares = sorted([(prop[i, j] + prop[j, i], i, j) for i in range(n_classes)
                 for j in range(i + 1, n_classes)], reverse=True)[:3]
print("OS PARES QUE MAIS TROCAM:")
for s, i, j in _pares:
    lado = "simétrico (as classes se parecem)" if min(prop[i, j], prop[j, i]) > .3 * max(prop[i, j], prop[j, i]) \
           else "de um lado só (costuma ser desequilíbrio no número de fotos)"
    print(f"   {CLASSES[i]} ↔ {CLASSES[j]}: {prop[i, j]:.0%} e {prop[j, i]:.0%} — {lado}")
print()
print(classification_report(y_va, previsoes, labels=range(n_classes), target_names=CLASSES,
                            digits=3, zero_division=0))

In [ ]:
#@title 🔥 Onde o modelo olhou? O mapa de atenção { display-mode: "form" }
#@markdown Mostra o calor sobre algumas fotos da validação (acertos e erros).
quantas = 6  #@param {type:"slider", min:2, max:10, step:1}
modelo.eval()
_escolha = list(np.where(previsoes == y_va)[0][:quantas // 2]) + list(np.where(previsoes != y_va)[0][:quantas - quantas // 2])
fig, axes = plt.subplots(2, len(_escolha), figsize=(2.6 * len(_escolha), 5.4))
axes = np.array(axes).reshape(2, -1)
for k, pos in enumerate(_escolha):
    foto, real = base_va[int(idx_va[pos])]
    calor = mapa_de_atencao(modelo, modelo_escolhido, foto, int(previsoes[pos]))
    img = (foto.permute(1, 2, 0).numpy() * np.array(desvio) + np.array(media)).clip(0, 1)
    cor = "#2F9E44" if previsoes[pos] == real else "#B00020"
    axes[0, k].imshow(img); axes[0, k].axis("off")
    axes[0, k].set_title(f"era {CLASSES[real]}\ndisse {CLASSES[previsoes[pos]]}", color=cor, fontsize=9)
    axes[1, k].imshow(img); axes[1, k].imshow(calor, cmap="jet", alpha=.45); axes[1, k].axis("off")
modelo.zero_grad()
plt.suptitle("Em cima a foto; embaixo, onde o modelo olhou para decidir", fontweight="bold")
plt.tight_layout(); plt.show()
print("Dois modelos com o mesmo F1 podem olhar para lugares diferentes. Um olha o objeto e")
print("continua acertando com outra luz e outra câmera; o outro olha o FUNDO e desaba quando")
print("o fundo muda. Nenhuma métrica separa os dois: só olhar separa.")

In [ ]:
#@title 😬 Ele erra com confiança { display-mode: "form" }
conf = prob_va.max(axis=1)
errou = previsoes != y_va
print(f"Confiança média nos ACERTOS ... {conf[~errou].mean()*100:5.1f}%")
print(f"Confiança média nos ERROS ..... {conf[errou].mean()*100 if errou.any() else float('nan'):5.1f}%")
if errou.any():
    _pior = np.where(errou)[0][np.argmax(conf[errou])]
    print(f"\nO erro mais confiante: era {CLASSES[y_va[_pior]]}, disse {CLASSES[previsoes[_pior]]} "
          f"com {conf[_pior]*100:.0f}% de 'certeza'.")
print("\nEsse número é uma conta interna, não uma garantia. Um modelo diz '97% de certeza'")
print("mesmo quando está errado: por isso a confiança sozinha não pode decidir nada.")

In [ ]:
#@title 👥 A média engana: fotos escuras × fotos claras { display-mode: "form" }
_brilho = np.array([np.asarray(Image.open(base_va.samples[int(i)][0]).convert("L")).mean()
                    for i in idx_va])
_corte = np.median(_brilho)
for nome_g, m in [("fotos mais ESCURAS", _brilho <= _corte), ("fotos mais CLARAS", _brilho > _corte)]:
    print(f"   {nome_g:<20} {m.sum():>4} fotos · F1-macro {f1_score(y_va[m], previsoes[m], average='macro')*100:5.1f}%")
print("\nSe um grupo vai bem pior, quem for atendido nas condições desse grupo recebe um")
print("sistema pior — e a média geral não conta isso.")

In [ ]:
#@title ⏱️ Custo contra ganho: os experimentos desta sessão { display-mode: "form" }
display(pd.DataFrame(registro))
print("A EfficientNet é a mais barata; a ViT, ~4× mais cara. Um ponto a mais de F1 vale")
print("quantos minutos de GPU? A resposta honesta depende do tempo que sobrou.")

---
# 🤔 PARADA OBRIGATÓRIA: o modelo que vocês usaram não é de vocês

> **Rosa:** *"Quem ensinou esse modelo a enxergar? Com que fotos?"*

1. **Ele herda o que estava torto.** O pré-treino viu ~1 milhão de fotos escolhidas por
   outra equipe, em outro país. O ajuste fino de vocês por cima não apaga essa inclinação.
   Que tipo de material ou de foto pode estar sub-representado?
2. **Ele erra com confiança.** A célula de confiança mostrou o erro mais "certo". Numa
   esteira, um modelo que diz 97% e erra deveria decidir sozinho?
3. **Ele pode acertar pelo motivo errado.** Todas as fotos têm o mesmo fundo. O mapa de
   atenção mostrou o calor sobre o material ou sobre o fundo?
4. **IA generativa.** Podem usar, desde que saibam explicar o que ela entregou: registrem
   qual, para quê, como conferiram e o que mudaram.

In [ ]:
#@title 📝 O relatório da sua equipe { display-mode: "form" }
nome_da_equipe = ""  #@param {type:"string"}
#@markdown **1. Que modelo e que valores dos quatro botões vocês usaram? Que experimentos fizeram?**
resposta_1 = ""  #@param {type:"string"}
#@markdown **2. O que as curvas e a matriz mostraram? Quais pares mais trocam, e por quê?**
resposta_2 = ""  #@param {type:"string"}
#@markdown **3. Onde o modelo olhou (mapa de atenção)? Ele acerta pelo motivo certo?**
resposta_3 = ""  #@param {type:"string"}
#@markdown **4. O modelo é emprestado: que bagagem ele traz, e em que condições da esteira ele deixa de valer?**
resposta_4 = ""  #@param {type:"string"}
#@markdown **5. Usaram IA generativa? Qual, para quê, como conferiram e o que mudaram?**
resposta_5 = ""  #@param {type:"string"}

relatorio = {"equipe": nome_da_equipe, "modelo": modelo_escolhido,
             "taxa_estagio1": taxa_estagio1, "epocas_estagio1": epocas_estagio1,
             "taxa_estagio2": taxa_estagio2, "epocas_estagio2": epocas_estagio2,
             "lote": lote, "aumento": aumento, "melhor_epoca": estado["melhor_epoca"],
             "validacao_pct": tamanho_da_validacao, "piso": round(piso, 4),
             "f1_macro": round(f1, 4), "acuracia": round(acuracia, 4),
             "just_configuracao": resposta_1, "just_curvas_e_matriz": resposta_2,
             "just_atencao": resposta_3, "just_etica": resposta_4,
             "uso_ia_generativa": resposta_5}
faltando = [k for k in ["equipe", "just_configuracao", "just_curvas_e_matriz", "just_atencao",
                        "just_etica", "uso_ia_generativa"] if not str(relatorio[k]).strip()]
print("⚠️ Ainda faltam campos: " + ", ".join(faltando) if faltando
      else "Relatório completo. Prossigam para a submissão.")

In [ ]:
#@title 🚀 Gerar o arquivo de submissão { display-mode: "form" }
import json, pathlib
PASTA_SUBMISSAO = pathlib.Path("submissoes")
PASTA_SUBMISSAO.mkdir(parents=True, exist_ok=True)
if "relatorio" not in dir() or not isinstance(relatorio, dict):
    relatorio = {"equipe": "", "modelo": modelo_escolhido, "f1_macro": round(f1, 4)}

# O teste passa pelo preparo da VALIDAÇÃO: o mundo como ele é, sem aumento.
modelo.eval()
palpites = []
with torch.no_grad():
    for k in range(0, N_TESTE, lote):
        fotos = torch.stack([preparo_val(Image.open(f).convert("RGB"))
                             for f in FOTOS_TESTE[k:k + lote]]).to(DISPOSITIVO)
        palpites += modelo(fotos).argmax(dim=1).cpu().tolist()
submissao = pd.DataFrame({"id_foto": IDS_TESTE, "classe_prevista": [CLASSES[p] for p in palpites]})

assert len(submissao) == N_TESTE, f"A submissão precisa ter {N_TESTE} linhas."
csv_saida = PASTA_SUBMISSAO / "submissao_recomeco.csv"
json_saida = PASTA_SUBMISSAO / "relatorio_recomeco.json"
submissao.to_csv(csv_saida, index=False)
with open(json_saida, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

print("="*60)
print("             ARQUIVOS PRONTOS PARA ENVIO")
print("="*60)
print(f"  submissao_recomeco.csv   ({len(submissao)} linhas)")
print("  relatorio_recomeco.json")
print("="*60)
print("\n  Distribuição prevista no teste:")
for k, v in submissao.classe_prevista.value_counts().items():
    print(f"     {EMOJI[k]} {k:<14} {v:>4}")
display(submissao.head())
print("\n  Para corrigir (da raiz do repositório):")
print(f"     python simulados/corrigir.py recomeco {csv_saida}")

try:
    from google.colab import files
    files.download(str(csv_saida))
    files.download(str(json_saida))
    print("\n  Download iniciado.")
except ImportError:
    print(f"\n  Fora do Colab: os arquivos estão em {PASTA_SUBMISSAO.resolve()}")

---
# 🌟 FIM DA MISSÃO RECOMEÇO · FASE 2

### Competências praticadas

- Transferência de aprendizado: por que funciona com poucas fotos
- Resolução de entrada e o preparo que vem com os pesos
- Aumento de dados só no treino (e como detectar quando vaza)
- Dois estágios: sondagem da saída e ajuste fino do último bloco
- O nome da camada de saída de cada arquitetura
- Curvas de imagem, matriz de confusão, mapa de atenção e confiança
- A bagagem de um modelo emprestado